In [2]:
import pandas as pd
import geopandas as gpd
import numpy as np
import rasterio
import catboost
import shap

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
from catboost import CatBoostRegressor
from sklearn.linear_model import LinearRegression

import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from pathlib import Path

In [3]:
REPO_ROOT = Path.cwd().parent
LFS_DTA = (
    REPO_ROOT
    / "ERF_Data/Data"
    / "Labor Force Survey, LFS 2022 - Egypt, Arab Rep., 2022"
    / "Egypt 2022-LFS IND-V1.dta"
)
HH_DTA = (
    REPO_ROOT
    / "ERF_Data/Data"
    / "Labor Force Survey, LFS 2022 - Egypt, Arab Rep., 2022"
    / "Egypt 2022-LFS HH-V1.dta"
)
CROSSWALK_CSV = (
    REPO_ROOT / "ERF_Data/Data" / "crosswalk" / "reg_governorate_crosswalk.csv"
)
COMMAND_AREA_CROSSWALK_CSV = (
    REPO_ROOT / "ERF_Data/Data" / "crosswalk" / "command_area_governorate_crosswalk.csv"
)
SPATIAL_XLSX = REPO_ROOT / "Data" / "preprocessed_data_explain.xlsx"
OUTPUT_CSV = Path("lfs2022_individual_preprocessed.csv")

WEEKS_PER_MONTH = 365.25 / 7 / 12  # ~4.348
MIN_COMMAND_AREA_COVERAGE = 0.05
MIN_WORKING_AGE = 15

IND_LABELS = {
    10: "Agriculture, forestry and fishing",
    20: "Mining and quarrying",
    30: "Manufacturing",
    40: "Electricity, gas and water supply",
    50: "Construction",
    60: "Wholesale and retail trade",
    70: "Transportation and storage",
    80: "Accommodation and food service activities",
    90: "Information and communication",
    100: "Financial and insurance activities",
    110: "Real estate, professional and support service activities",
    120: "Public administration and defense",
    130: "Education",
    140: "Human health and social work activities",
    150: "Other activities",
}
EMPS_LABELS = {
    1: "Employee",
    2: "Employer",
    3: "Own-account/self-employed",
    4: "Unpaid family worker",
    5: "Producers cooperative",
    6: "Not classifiable",
}
EDUC_LABELS = {
    1: "None",
    2: "Primary/Lower secondary",
    3: "Secondary",
    4: "Post secondary or equivalent",
    5: "University",
    6: "Postgraduate",
}
EMPSTAB_LABELS = {
    1: "Full time/Regular",
    2: "Part time/Temporary",
    3: "Seasonal/Irregular",
}
REL_LABELS = {
    1: "Head",
    2: "Spouse",
    3: "Son/daughter",
    4: "Parent",
    5: "Sibling",
    6: "Grandchild",
    7: "Other relative",
    8: "Non-relative",
}
OCC_LABELS = {
    10: "Managers",
    20: "Professionals",
    30: "Technicians",
    40: "Clerks",
    50: "Service/sales",
    60: "Skilled agricultural/fishery",
    70: "Craft workers",
    80: "Plant/machine operators",
    90: "Elementary occupations",
    100: "Armed forces",
    998: "Other",
    999: "Not stated",
}
# Standard ERF-harmonized marital-status scheme (code 1 confirmed as "Never married" in the
# dictionary; 2-5 follow the standard ILO/ERF ordering, not independently re-verified against a
# complete codebook; the source dictionary's value list is truncated for this variable).
MART_LABELS = {
    1: "Never married",
    2: "Married",
    3: "Widowed",
    4: "Divorced",
    5: "Separated",
}
YES_NO = {0: "No", 1: "Yes"}

In [7]:
df = pd.read_stata(LFS_DTA, convert_categoricals=False)
hh = pd.read_stata(HH_DTA, convert_categoricals=False)
print("individual shape:", df.shape)
print("household shape:", hh.shape)

individual shape: (299423, 106)
household shape: (76976, 116)


In [8]:
household_size = df.groupby("caseser").size()
n_employed_in_hh = df[df["emps"].notna()].groupby("caseser").size()
head_info = (
    df[df["rel"] == 1].drop_duplicates("caseser").set_index("caseser")[["educ", "sex"]]
)
head_info.columns = ["head_educ", "head_sex"]

print("household_size: computed for", len(household_size), "households")
print(
    "n_employed_in_hh: computed for",
    len(n_employed_in_hh),
    "households (that have >=1 employed member)",
)
print("head_info: found a head for", len(head_info), "households")
print(
    "hh file: malinlf/feminlf/occhd available for",
    hh["caseser"].nunique(),
    "households",
)

household_size: computed for 76976 households
n_employed_in_hh: computed for 57695 households (that have >=1 employed member)
head_info: found a head for 76976 households
hh file: malinlf/feminlf/occhd available for 76976 households


In [9]:
household_size = df.groupby("caseser").size()
n_employed_in_hh = df[df["emps"].notna()].groupby("caseser").size()
head_info = (
    df[df["rel"] == 1].drop_duplicates("caseser").set_index("caseser")[["educ", "sex"]]
)
head_info.columns = ["head_educ", "head_sex"]

print("household_size: computed for", len(household_size), "households")
print(
    "n_employed_in_hh: computed for",
    len(n_employed_in_hh),
    "households (that have >=1 employed member)",
)
print("head_info: found a head for", len(head_info), "households")
print(
    "hh file: malinlf/feminlf/occhd available for",
    hh["caseser"].nunique(),
    "households",
)

household_size: computed for 76976 households
n_employed_in_hh: computed for 57695 households (that have >=1 employed member)
head_info: found a head for 76976 households
hh file: malinlf/feminlf/occhd available for 76976 households


In [10]:
avg_hours_by_ind = df[df["hrswk"] > 0].groupby("ind")["hrswk"].mean()

head_hours = df[df["rel"] == 1].drop_duplicates("caseser").set_index("caseser")["hrswk"]
spouses_ind = df[df["rel"] == 2].sort_values(["caseser", "pnum"]).copy()
spouses_ind["spouse_order"] = spouses_ind.groupby("caseser").cumcount() + 1
spouse_hours = {
    n: spouses_ind[spouses_ind["spouse_order"] == n].set_index("caseser")["hrswk"]
    for n in [1, 2, 3]
}


def convert_irregular(daily_wage, ind_code, real_hours, label):
    used_real = real_hours.notna() & (real_hours > 0)
    hours = real_hours.where(used_real, ind_code.map(avg_hours_by_ind))
    n_earners = (daily_wage > 0).sum()
    n_real = (used_real & (daily_wage > 0)).sum()
    if n_earners:
        print(
            f"  {label}: {n_real}/{n_earners} irregular earners matched to real individual hours "
            f"({n_real/n_earners*100:.1f}%), rest used industry average"
        )
    workdays_per_week = hours / 8
    return daily_wage * workdays_per_week * 4


hh_wage = hh[hh["rururb"] == 0].copy()
hh_wage["head_hrswk"] = hh_wage["caseser"].map(head_hours)
print(
    "Irregular-wage-to-monthly conversion, real-hours match rate (rural households only):"
)
hh_wage["irrwagehd"] = convert_irregular(
    hh_wage["irrgwaghd"].fillna(0), hh_wage["indhd"], hh_wage["head_hrswk"], "head"
)
for n in [1, 2, 3]:
    hh_wage[f"sp_{n}_hrswk"] = hh_wage["caseser"].map(spouse_hours[n])
    hh_wage[f"irrwagesp_{n}"] = convert_irregular(
        hh_wage[f"irrgwagsp_{n}"].fillna(0),
        hh_wage[f"indsp_{n}"],
        hh_wage[f"sp_{n}_hrswk"],
        f"spouse_{n}",
    )

hh_wage["wagehd"] = hh_wage[["empinchd", "sempinchd", "totwaghd", "irrwagehd"]].sum(
    axis=1, skipna=True
)
sp_cols = []
for n in [1, 2, 3]:
    sp_cols += [f"empincsp_{n}", f"sempincsp_{n}", f"totwagsp_{n}", f"irrwagesp_{n}"]
hh_wage["wagesp"] = hh_wage[sp_cols].sum(axis=1, skipna=True)
hh_wage["household_wage_total"] = hh_wage["wagehd"] + hh_wage["wagesp"]
hh_wage["wage_ratio"] = (hh_wage["wagesp"] / hh_wage["household_wage_total"]).fillna(
    0.5
)

print()
print(hh_wage[["household_wage_total", "wage_ratio"]].describe())

Irregular-wage-to-monthly conversion, real-hours match rate (rural households only):
  head: 7511/7538 irregular earners matched to real individual hours (99.6%), rest used industry average
  spouse_1: 195/200 irregular earners matched to real individual hours (97.5%), rest used industry average

       household_wage_total    wage_ratio
count          45426.000000  45426.000000
mean            2209.675539      0.190799
std             3359.425282      0.252778
min                0.000000      0.000000
25%                0.000000      0.000000
50%             2250.000000      0.000000
75%             3200.000000      0.500000
max           150000.000000      1.000000


In [11]:
n_underage = (df["age"] < MIN_WORKING_AGE).sum()
working_age = df[df["age"] >= MIN_WORKING_AGE].copy()
print(
    f"Dropped {n_underage} respondents under age {MIN_WORKING_AGE} (working-age floor)"
)

survey_df = df[df["emps"].notna()].copy()
print("shape after employed filter:", survey_df.shape)
print(survey_df["emps"].map(EMPS_LABELS).value_counts())

Dropped 100476 respondents under age 15 (working-age floor)
shape after employed filter: (76411, 106)
emps
Employee                     56065
Own-account/self-employed    14188
Unpaid family worker          3696
Employer                      2367
Name: count, dtype: int64


In [19]:
src_dir = (
    Path("~").expanduser()
    / "OneDrive - Stichting Deltares/PhD/Egypt/04_Data/2026_data/"
)
excel_path = (
    Path("~").expanduser()
    / "OneDrive - Stichting Deltares/PhD/Egypt/data_correlation.xlsx"
)

In [20]:
masking_df = pd.read_excel(
    src_dir.parent / "Result/correlation.xlsx", sheet_name="Sheet1"
)

distribution_df = pd.read_csv(
    src_dir.parent / "Result/population_density_matrix.csv", index_col="NAME1_"
)
excel_df = pd.read_excel(excel_path, sheet_name="command_unit")

In [21]:
survey_df['sempinc'] = survey_df['sempinc'].fillna(0)
survey_df['irrgwag'] = survey_df['irrgwag'].fillna(0)
survey_df['totwag'] = survey_df['totwag'].fillna(0)

survey_df['totwag'] = survey_df['totwag'] + survey_df['sempinc'] + survey_df['irrgwag'] * 30

In [24]:
to_correlate = masking_df.loc[masking_df["correl"] == 1, "variable"]

crosswalk = pd.read_csv(CROSSWALK_CSV)

survey_df = survey_df.merge(
    crosswalk[["reg_code", "OBJECTID", "NAME1_"]],
    left_on="reg",
    right_on="reg_code",
    how="left",
)

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\hermawan\\OneDrive - Stichting Deltares\\PhD\\Egypt_ERF_data\\egypt-survey-ml\\ERF_Data\\Data\\crosswalk\\reg_governorate_crosswalk.csv'

In [ ]:
governorates = survey_df[survey_df["NAME1_"].notna()]["NAME1_"].unique()

rng = np.random.default_rng(seed=42)
sample_df = survey_df.copy(deep=True)
sample_df["command_unit"] = None
command_units = distribution_df.columns

gov_indices = {
    gov: idx.to_numpy() for gov, idx in survey_df.groupby("NAME1_").groups.items()
}
# female_gov_indices = {
#     gov: idx.to_numpy() for gov, idx in female_df.groupby("NAME1_").groups.items()
# }
# male_gov_indices = {
#     gov: idx.to_numpy() for gov, idx in male_df.groupby("NAME1_").groups.items()
# }

for gov, idx in gov_indices.items():
    probs = distribution_df.loc[gov].to_numpy()

    assignments = rng.choice(
        command_units,
        size=len(idx),
        p=probs,
    )

    sample_df.loc[idx, "command_unit"] = assignments

sample_df.loc[sample_df["command_unit"] == "other", "command_unit"] = None

sample_df = sample_df[sample_df["command_unit"].notna()]
final_sample = sample_df.merge(
    excel_df, left_on="command_unit", right_on="area", suffixes=("", "_spatial")
)

to_correlate = masking_df.loc[masking_df["correl"] == 1, "variable"]
final_sample = final_sample.dropna(subset=['mart_d', 'yeduc'])
final_sample['hrswksc'] = final_sample['hrswksc'].fillna(0)
final_sample['hrswk'] = final_sample['hrswk'] + final_sample['hrswksc']

final_sample['immigr'] = final_sample['immigr'].fillna(3)
final_sample['disabl'] = final_sample['disabl'].fillna(0)

final_sample = final_sample.drop(columns=['unempdur', 'numwrk'])
cols = [c for c in to_correlate if c in final_sample.columns]

In [ ]:
categorical_columns = masking_df[(masking_df["type"] == 'binary') | (masking_df["type"] == 'categorical') & (masking_df["variable"].isin(cols))]['variable'].values.tolist()
final_sample[categorical_columns] = final_sample[categorical_columns].astype(str)

In [ ]:
final_sample_female = final_sample[final_sample['sex'] == '2']
# final_sample_female = final_sample_female[final_sample_female['hrswk'] > 0]
final_sample_female = final_sample_female[final_sample_female['totwag'] > 0]
final_sample_female = final_sample_female[final_sample_female['totwag'] < 10000]

X = final_sample_female[cols].drop(columns=["hrswk", "hrswksc", "totwag"])
# y = final_sample_female["hrswk"]
y = final_sample_female["totwag"]
# y = np.log1p(final_sample_female["totwag"])

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

In [ ]:
for depth in [4, 6, 8, 10, 12]:
    for lr in [0.01, 0.03, 0.1]:
        model = CatBoostRegressor(
            iterations=5000,
            learning_rate=lr,
            depth=depth,
            loss_function="RMSE",
            random_seed=42,
            verbose=False,
        )
    
        model.fit(
            X_train,
            y_train,
            eval_set=(X_val, y_val),
            cat_features=categorical_columns,
            use_best_model=True,
            early_stopping_rounds=200,
        )
    
        pred = model.predict(X_test)
    
        print(
            f"depth={depth:2d}, "
            f"lr={lr}, "
            f"R²={r2_score(y_test, pred):.3f}, "
            f"best_iter={model.get_best_iteration()}"
        )

In [ ]:
final_sample_female = final_sample[final_sample['sex'] == '2']
# final_sample_female = final_sample_female[final_sample_female['hrswk'] > 0]
final_sample_female = final_sample_female[final_sample_female['totwag'] > 0]
final_sample_female = final_sample_female[final_sample_female['totwag'] < 10000]

X = final_sample_female[cols].drop(columns=["hrswk", "hrswksc", "totwag"])
# y = final_sample_female["hrswk"]
y = final_sample_female["totwag"]
# y = final_sample_female[final_sample_female["totwag"] < 10000] 
# y = np.log1p(final_sample_female["totwag"])

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)


# Use depth and learning rate with best performance
model = CatBoostRegressor(
    iterations=5000,
    learning_rate=0.03,
    depth=6,
    loss_function="RMSE",
    eval_metric="RMSE",
    random_seed=42,
    verbose=100,
)



model.fit(
    X_train,
    y_train,
    eval_set=(X_val, y_val),
    cat_features=categorical_columns,
    use_best_model=True,
    early_stopping_rounds=200,
)

# predictions on unseen data
y_pred = model.predict(X_test)
y_pred_wage = np.expm1(y_pred)
y_test_wage = np.expm1(y_test)
print(model.get_best_iteration())


# print("R²:", r2_score(y_test, y_pred))
# print("MAE (€):", mean_absolute_error(y_test_wage, y_pred_wage))
# print("RMSE (€):", root_mean_squared_error(y_test_wage, y_pred_wage))

print("R²:", r2_score(y_test, y_pred))
print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", root_mean_squared_error(y_test, y_pred))

In [ ]:
agri_female = final_sample_female[final_sample_female['ind'] == 10]
X_agri = agri_female[cols].drop(columns=["hrswk", "hrswksc", "totwag"])
y_agri = agri_female["totwag"]

y_agri_pred = model.predict(X_agri)

print("R² (agri):", r2_score(y_agri, y_agri_pred))
print("MAE (agri):", mean_absolute_error(y_agri, y_agri_pred))
print("RMSE (agri):", root_mean_squared_error(y_agri, y_agri_pred))

In [ ]:
y.hist(bins=50)

In [ ]:
pd.Series(
    model.get_feature_importance(),
    index=X.columns
).sort_values(ascending=False).head(20)

In [ ]:
# Simple linear regression for comparison
lr = LinearRegression()
lr.fit(pd.get_dummies(X, drop_first=True), y)

pred = lr.predict(pd.get_dummies(X, drop_first=True))

print("R² (linear model):", r2_score(y, pred))

In [ ]:
# Comparison to just taking the mean
baseline = np.repeat(y_train.mean(), len(y_test))

print("R²:", r2_score(y_test, baseline))
print("Baseline RMSE:", root_mean_squared_error(y_test, baseline))
print("Model RMSE:", root_mean_squared_error(y_test, y_pred))

In [ ]:
plt.scatter(y_test, y_pred, alpha=0.3)
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    "r--"
)

plt.xlabel("Observed wage")
plt.ylabel("Predicted wage")
plt.show()

In [ ]:
imp = (
    pd.Series(
        model.feature_importances_,
        index=X_train.columns
    )
    .sort_values(ascending=False)
)

print(imp.head(20))

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

shap.summary_plot(shap_values, X_test)

In [ ]:
# from catboost import Pool, cv

# train_pool = Pool(
#     X,
#     y,
#     cat_features=categorical_columns,
# )

# params = {
#     "loss_function": "RMSE",
#     "depth": 8,
#     "learning_rate": 0.03,
#     "random_seed": 42,
# }

# cv_results = cv(
#     train_pool,
#     params,
#     fold_count=5,
#     iterations=5000,
#     early_stopping_rounds=200,
# )

# print(cv_results.tail())